# 00 - Single-image inspection walkthrough

Start here. This notebook needs one local image only: no `sources.csv`, manifest, split, or prepared dataset. It previews every transformation in memory. Detection, masks, toplines, and sagitta are inspection measurements—not posture ground-truth labels.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
print("project root:", PROJECT_ROOT)

## Configuration

With the defaults below, nothing is written. Use `MODEL_NAME="none"` when `IMAGE_PATH` already points to a cow crop. For a multi-cow image, first run the detection panel, then set `DETECTION_INDEX` to one of its visible `[index]` labels.

In [ ]:
IMAGE_PATH = PROJECT_ROOT / "data" / "inbox" / "one_cow.jpg"
MODEL_NAME = "yolo11n-seg.pt"
PREVIEW_ONLY = True
SAVE_OUTPUTS = False

DETECTION_INDEX = None  # required only when the detector finds multiple cows
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "single_image_inspection"

if not IMAGE_PATH.exists():
    raise FileNotFoundError(
        f"Input image not found: {IMAGE_PATH}. Put one cow image at "
        f"{PROJECT_ROOT / 'data' / 'inbox' / 'one_cow.jpg'} or change IMAGE_PATH."
    )

## 1–2. Raw image and indexed cow detections

Every detection is drawn and indexed. A detector without segmentation masks is still useful here; the crop panels remain available.

In [ ]:
import cv2

from cowarch.detect import load_detector
from cowarch.prepare import PrepareConfig, blank_record, detect_frame_cows, process_frame
from cowarch.viz import _show, panel_detections

frame = cv2.imread(str(IMAGE_PATH))
if frame is None:
    raise RuntimeError(f"OpenCV could not decode image: {IMAGE_PATH}")

resolved_model_name = MODEL_NAME
if str(MODEL_NAME).lower() != "none":
    requested = Path(MODEL_NAME).expanduser()
    candidates = [requested] if requested.is_absolute() else [PROJECT_ROOT / requested, Path.cwd() / requested]
    local_checkpoint = next((path.resolve() for path in candidates if path.exists()), None)
    if local_checkpoint is None and (PREVIEW_ONLY or not SAVE_OUTPUTS):
        raise FileNotFoundError(
            f"Checkpoint {MODEL_NAME!r} is not available locally. Preview mode refuses "
            "Ultralytics' implicit weight download because the default run must not write. "
            "Place the checkpoint under PROJECT_ROOT, set MODEL_NAME to its local path, "
            "or use MODEL_NAME='none' for a ready crop."
        )
    if local_checkpoint is not None:
        resolved_model_name = str(local_checkpoint)

detector, cow_class = load_detector(resolved_model_name)
config = PrepareConfig(
    confidence=0.35, padding=0.03, min_area_ratio=0.0, max_area_ratio=1.0,
    hard_side_filter=False, reject_border=False, dedup_hamming=0,
)
detections = detect_frame_cows(
    frame, detector=detector, cow_class=cow_class, confidence=config.confidence
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
_show(axes[0], frame, "1. raw image")
panel_detections(axes[1], frame, detections, box=None)
fig.tight_layout(); plt.show()
print(f"detected cows: {len(detections)}; no candidate has been rejected in this preview")

In [ ]:
n_detections = len(detections)
if n_detections == 0:
    raise RuntimeError("No cow was detected. Lower confidence, choose another checkpoint, or use MODEL_NAME='none' for a ready crop.")
if n_detections > 1 and DETECTION_INDEX is None:
    raise ValueError(
        f"Found {n_detections} cows. Set DETECTION_INDEX to a displayed index "
        "and rerun this cell; preview does not auto-reject or guess the animal."
    )
selected_index = 0 if DETECTION_INDEX is None else DETECTION_INDEX
record = blank_record("single_image", IMAGE_PATH.stem, 0, IMAGE_PATH.name)
outcome = process_frame(
    frame, record, detector=detector, cow_class=cow_class, config=config,
    detection_index=selected_index,
)
print(f"selected detection: {selected_index}; accepted for inspection: {outcome.accepted}")

## 3–6. Crop, mask, and two silhouette baselines

Panel 5 keeps the full silhouette topline. Panel 6 is the fixed 20% trim and is explicitly an **experimental silhouette baseline**; it is not anatomically anchored.

In [ ]:
from cowarch.viz import panel_mask, panel_topline

inspection_figure, axes = plt.subplots(1, 4, figsize=(18, 4.5))
_show(axes[0], outcome.crop, "3. selected cow crop")
panel_mask(axes[1], outcome.crop, outcome.mask)
panel_topline(axes[2], outcome.crop, outcome.mask, trim=0.0)
axes[2].set_title("5. untrimmed silhouette topline", fontsize=9)
panel_topline(axes[3], outcome.crop, outcome.mask, trim=0.20)
axes[3].set_title("6. experimental silhouette baseline (20% trim)", fontsize=9)
inspection_figure.tight_layout(); plt.show()
if outcome.mask is None:
    print("WARNING: this checkpoint returned no segmentation mask. Detection and crop remain inspectable; panels 5–8 are unavailable.")

## 7. Click anatomical anchors

Click exactly twice on the crop: **withers**, then **sacrum**. The next panel samples the dense mask topline only over the chord span. If clicks do not register, install `ipympl`, restart the kernel, and rerun this cell.

In [ ]:
%matplotlib widget
anchors = []
anchor_figure = None

if outcome.mask is None:
    print("No mask: anchor-based dense-contour inspection is unavailable.")
else:
    anchor_figure, anchor_ax = plt.subplots(figsize=(10, 5))
    _show(anchor_ax, outcome.crop, "7. click withers, then sacrum")

    def capture_anchor(event):
        if event.inaxes is not anchor_ax or event.xdata is None or len(anchors) >= 2:
            return
        anchors.append([float(event.xdata), float(event.ydata)])
        name = ("withers", "sacrum")[len(anchors) - 1]
        anchor_ax.plot(event.xdata, event.ydata, "o", color="#f4a259")
        anchor_ax.annotate(name, (event.xdata, event.ydata), color="white", xytext=(4, 5), textcoords="offset points")
        anchor_figure.canvas.draw_idle()
        print(f"captured {name}: ({event.xdata:.1f}, {event.ydata:.1f})")

    anchor_figure.canvas.mpl_connect("button_press_event", capture_anchor)
    plt.show()

## 8. Anatomical chord, signed sagitta, peak, and feature summary

Positive signed sagitta means dorsal/upward arch relative to the canonical left-to-right chord; negative means downward sag. Reflection must not change that meaning. These are features for inspection and modeling, never labels.

In [ ]:
%matplotlib inline
from cowarch.geometry import anchored_topline, anchored_topline_features

anchored_features = None
geometry_figure = None
if outcome.mask is None:
    print("No mask: no anchored features were computed.")
elif len(anchors) != 2:
    print(f"Captured {len(anchors)} anchor(s). Click withers and sacrum in the previous cell, then rerun this cell.")
else:
    anchor_array = np.asarray(anchors, dtype=float)
    anchored_features = anchored_topline_features(outcome.mask, anchor_array)

    dense = anchored_topline(outcome.mask, anchor_array)
    chord_start, chord_end = dense[0], dense[-1]
    chord_vector = chord_end - chord_start
    chord_length = float(np.linalg.norm(chord_vector))
    chord_unit = chord_vector / chord_length
    peak_index = int(round(anchored_features["anchored_peak_position"] * (len(dense) - 1)))
    peak = dense[peak_index]
    peak_foot = chord_start + ((peak - chord_start) @ chord_unit) * chord_unit

    geometry_figure, axes = plt.subplots(1, 2, figsize=(15, 5))
    _show(axes[0], outcome.crop, "7. dense anchored topline")
    axes[0].plot(dense[:, 0], dense[:, 1], color="#f4a259", linewidth=2, label="smoothed dense topline")
    axes[0].plot(anchor_array[:, 0], anchor_array[:, 1], "o", color="white", label="withers / sacrum")
    axes[0].legend(fontsize=8)

    _show(axes[1], outcome.crop, "8. chord, signed sagitta, and peak")
    axes[1].plot([chord_start[0], chord_end[0]], [chord_start[1], chord_end[1]], "--", color="white", label="anatomical chord")
    axes[1].plot(dense[:, 0], dense[:, 1], color="#f4a259", linewidth=2)
    axes[1].plot(peak[0], peak[1], "o", color="#d1495b", label="peak")
    axes[1].plot([peak[0], peak_foot[0]], [peak[1], peak_foot[1]], color="#d1495b", linewidth=2, label="signed sagitta")
    axes[1].legend(fontsize=8)
    geometry_figure.tight_layout(); plt.show()
    display(pd.Series(anchored_features, name="inspection feature (not a label)").to_frame())

## Optional save

This is the only writing cell. It runs only when `PREVIEW_ONLY=False` **and** `SAVE_OUTPUTS=True`. `data/` and `outputs/` stay gitignored.

In [ ]:
import json

if PREVIEW_ONLY or not SAVE_OUTPUTS:
    print("PREVIEW MODE - no files written. Both PREVIEW_ONLY=False and SAVE_OUTPUTS=True are required.")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    inspection_figure.savefig(OUTPUT_DIR / f"{IMAGE_PATH.stem}_inspection.png", bbox_inches="tight")
    if geometry_figure is not None and anchored_features is not None:
        geometry_figure.savefig(OUTPUT_DIR / f"{IMAGE_PATH.stem}_anchored_geometry.png", bbox_inches="tight")
        (OUTPUT_DIR / f"{IMAGE_PATH.stem}_anchored_features.json").write_text(
            json.dumps(anchored_features, indent=2), encoding="utf-8"
        )
    print("wrote inspection outputs to", OUTPUT_DIR)